<a href="https://colab.research.google.com/github/muneerh-fa/Sdaia-agentic-ai-systems-course/blob/main/assignments/assignment4/assignment4_semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4 — Semantic Search

This notebook applies semantic search using my own PDF documents about visual pollution prediction and deep learning.

## Overview

This notebook follows the semantic search pipeline:

1. Load documents
2. Split documents into chunks
3. Generate embeddings
4. Store embeddings in a vector store
5. Retrieve the most relevant chunks for a query

## Import Libraries

In [2]:
%pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters pypdf sentence-transformers

In [3]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

## Specify the PDF Files



In [4]:
pdf_files = [
    "Smart_Coastline_Environment_Management_Using_Deep_Detection_of_Manmade_Pollution_and_Hazards.pdf",
    "Building an automatic system.pdf",
    "vpp.pdf",
]

## Load the PDF Documents

In [5]:
docs = []

for file_path in pdf_files:
    loader = PyPDFLoader(file_path)
    file_docs = loader.load()
    docs.extend(file_docs)

print("Total loaded pages:", len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:500])

Total loaded pages: 68
{'producer': 'Microsoft® Word 2016; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': "'Certified by IEEE PDFeXpress at 02/26/2019 8:03:47 PM'", 'creationdate': '2019-02-27T07:25:52+03:30', 'application': "'Certified by IEEE PDFeXpress at 02/26/2019 8:03:47 PM'", 'ieee issue id': '8734902', 'ieee article id': '8735012', 'title': 'Smart Coastline Environment Management Using Deep Detection of Manmade Pollution and Hazards', 'ieee publication id': '8732269', 'meeting ending date': '1 March 2019', 'meeting starting date': '28 Feb. 2019', 'subject': '2019 5th Conference on Knowledge Based Engineering and Innovation (KBEI);2019; ; ;', 'moddate': '2019-06-11T08:23:40-04:00', 'source': 'Smart_Coastline_Environment_Management_Using_Deep_Detection_of_Manmade_Pollution_and_Hazards.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
5th Conference on Knowledge-Based Engineering and Innovation, Iran University of Science and Technology, Tehr

## Split the Documents into Chunks

We split the pages into smaller chunks to improve retrieval quality.

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

print("Total chunks:", len(all_splits))
print(all_splits[0].metadata)
print(all_splits[0].page_content[:500])

Total chunks: 266
{'producer': 'Microsoft® Word 2016; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': "'Certified by IEEE PDFeXpress at 02/26/2019 8:03:47 PM'", 'creationdate': '2019-02-27T07:25:52+03:30', 'application': "'Certified by IEEE PDFeXpress at 02/26/2019 8:03:47 PM'", 'ieee issue id': '8734902', 'ieee article id': '8735012', 'title': 'Smart Coastline Environment Management Using Deep Detection of Manmade Pollution and Hazards', 'ieee publication id': '8732269', 'meeting ending date': '1 March 2019', 'meeting starting date': '28 Feb. 2019', 'subject': '2019 5th Conference on Knowledge Based Engineering and Innovation (KBEI);2019; ; ;', 'moddate': '2019-06-11T08:23:40-04:00', 'source': 'Smart_Coastline_Environment_Management_Using_Deep_Detection_of_Manmade_Pollution_and_Hazards.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'start_index': 0}
5th Conference on Knowledge-Based Engineering and Innovation, Iran University of Science and Tec

## Create Embeddings

This step converts text chunks into vectors that represent semantic meaning.

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Create the Vector Store

In [8]:
vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

print("Number of stored chunks:", len(ids))

Number of stored chunks: 266


## Perform Semantic Search

Now we retrieve the most relevant chunks based on the meaning of the query.

In [9]:
query = "How is deep learning used to detect visual pollution?"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:1000])


--- Result 1 ---
Source: Building an automatic system.pdf
Page: 8
Building an Automatic System for Detecting Manifestations of Visual Pollution Using 
Geospatial Techniques and Deep Learning: An Applied Study on Abandoned Vehicles in Riyadh 
15  
Arab International Journal of 
Information Technology & Data 
Vol. 4, No. 2 April – June 2024 
 
sensors is to provide real-time data on what is being monitored to ensure 
rapid intervention by governments and the concerned parties.  Moreover, 
sensors have proven to be effective in measuring and predicting we ather, 
traffic, and other environmental issues in the city. 
Artificial intelligence (AI) is one of the branches of computer 
science that uses software and machines to solve problems and make 
decisions, mimicking human intelligence (Ongsulee, 2018). Deep learning 
(DL) is considered a sub -field of artificial intelligence, as it relies on 
algorithms that are able to learn from unstructured data and complex 
patterns such as speech a

## Semantic Search with Scores

In [10]:
query = "What datasets or road images are used in visual pollution prediction?"

results_with_scores = vector_store.similarity_search_with_score(query, k=3)

for i, item in enumerate(results_with_scores, start=1):
    doc, score = item
    print(f"\n--- Result {i} ---")
    print("Score:", score)
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:1000])


--- Result 1 ---
Score: 0.6958272130330815
Source: vpp.pdf
Page: 6
MOMRAH VP dataset in-depth, and irrelevant, inaccurate, or unreliable images related 
to the visual pollution topic are immediately excluded. Some examples of irrelevant and 
excluded images are depicted in Figure 5. Since the normalization process could improve 
the overall prediction performance, the VP images are normalized to bring their intensity 
into the range of [0, 255] [13,18]. Meanwhile, all images are resized using bi-cubic interpo-
lation to scale their intensity pixels into the same range of 460 × 600. 
 
Figure 5. Some examples of the irrelevant images that are excluded during the pre-processing 
step. 
Figure 4. Visual pollution real dataset (i.e., the MOMRAH VP dataset) distribution over three different
classes: excavation barriers, potholes, and dilapidated sidewalks. The dataset per class is split into
70% for training, 10% for validation, and 20% for testing.
3.2. Data Pre-Processing
The following p

## Interpretation of the Results

The retrieved results show that the semantic search system was able to find relevant text chunks from the uploaded PDF articles based on the meaning of the query.

For example, when the query asked about how deep learning is used in visual pollution detection, the system returned chunks discussing:
- deep learning models
- object detection in images
- model evaluation
- visual pollution datasets

This demonstrates that the system searches by semantic similarity rather than exact keyword matching.

## Why This is Semantic Search

The system does not search only by exact keywords.
Instead, it compares vector embeddings to find text chunks with similar meaning to the user query.

## Conclusion

This notebook applied the semantic search pipeline to my own PDF documents about visual pollution prediction:

- Load
- Split
- Embed
- Store
- Retrieve

The retrieved results show how semantic search can find relevant information from long research documents based on meaning rather than exact wording.